# AI for Actuaries : Hackathon 2026

## 1. Summary
Team/Participant Name: \
Final Model Score in Public Leaderboard (RMSLE): \
One-line highlight of your approach: \

### Objective

Predict the expected ultimate net claim cost (risk premium) for each policy-period in 2025, net of policy excess and recoveries, excluding repudiated claims.

The modelling approach is based on:

[
\text{Expected Claim Cost} = \text{Exposure} \times \text{Frequency} \times \text{Severity}
]

## Week 1: Build Your Pricing Approach
At the start of the challenge, you will receive a synthetic insurance data from an established market. The data covers three lines of business: private motor, home buildings and home contents. It runs over four policy years (2021 to 2024) in a developed market.

Your task is to predict the expected ultimate claim cost for each policy in 2025 : the final cost of all claims with a loss date in the row’s period, net of the policy excess and net of recoveries, excluding declined claims.



In [21]:
# Install packages
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

# Models

#warnings.filterwarnings('ignore')

In [ ]:
#Read the datasets
from pathlib import Path

data_dir = Path("data")
if not data_dir.is_dir():
    raise FileNotFoundError(f"Data directory not found: {data_dir.resolve()}")

parquet_files = sorted(data_dir.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No Parquet datasets found in {data_dir.resolve()}")

datasets = {path.stem: pd.read_parquet(path) for path in parquet_files}

pd.DataFrame(
    [
        {"dataset": name, "rows": len(df), "columns": len(df.columns), "duplicate rows": df.duplicated().sum()}
        for name, df in datasets.items()
    ]
)

,dataset,rows,columns,duplicate rows
0,addresses,575983,16,0
1,claims,163150,12,0
2,customers,545240,18,0
3,policies_buildings,707726,27,0
4,policies_contents,721617,28,0
5,policies_motor,979968,39,0
6,sample_submission,626639,2,0
7,test_buildings,184155,26,0
8,test_contents,187498,27,0
9,test_motor,254986,38,0


## Data Preparation and Dataset Merging

Objective: Ensure that each row represents a valid policy-period and that reference tables have unique keys. \
Below the table we can see that the average exposure in the policies is more than 1 year.
Data quality is ok.

In [33]:
# STEP 2 — Validate unique keys and policy exposures

customers = datasets["customers"].copy()
addresses = datasets["addresses"].copy()
weather = datasets["weather"].copy()
claims = datasets["claims"].copy()

train = {
    line: datasets[f"policies_{line}"].copy()
    for line in ["motor", "buildings", "contents"]
}

test = {
    line: datasets[f"test_{line}"].copy()
    for line in ["motor", "buildings", "contents"]
}

# 1. Check unique keys
print("=== REFERENCE TABLE CHECKS ===")

for name, df, keys in [
    ("customers", customers, ["customer_id"]),
    ("addresses", addresses, ["address_id"]),
    ("weather", weather, ["address_id", "year"]),
    ("claims", claims, ["claim_id"])
]:
    duplicates = df.duplicated(subset=keys).sum()
    status = "PASS" if duplicates == 0 else "FAIL"
    print(f"{name:20} | {status} | Duplicate keys: {duplicates}")

# 2. Check policy-period structure
print("\n=== POLICY-PERIOD CHECKS ===")

results = []

for line in ["motor", "buildings", "contents"]:
    for dataset_type, df in [
        ("train", train[line]),
        ("test", test[line])
    ]:
        df["period_start"] = pd.to_datetime(df["period_start"])
        df["period_end"] = pd.to_datetime(df["period_end"])

        results.append({
            "dataset": f"{dataset_type}_{line}",
            "rows": len(df),
            "duplicate_row_ids": df["row_id"].duplicated().sum(),
            "invalid_dates": (
                df["period_start"] > df["period_end"]
            ).sum(),
            "missing_exposure": df["exposure"].isna().sum(),
            "nonpositive_exposure": (df["exposure"] <= 0).sum(),
            "total_exposure": df["exposure"].sum(),
            "average_exposure": df["exposure"].mean()
        })

quality_report = pd.DataFrame(results)

display(quality_report)

=== REFERENCE TABLE CHECKS ===
customers            | PASS | Duplicate keys: 0
addresses            | PASS | Duplicate keys: 0
weather              | PASS | Duplicate keys: 0
claims               | PASS | Duplicate keys: 0

=== POLICY-PERIOD CHECKS ===


,dataset,rows,duplicate_row_ids,invalid_dates,missing_exposure,nonpositive_exposure,total_exposure,average_exposure
0,train_motor,979968,0,0,0,0,1.058510e+06,1.080147
1,test_motor,254986,0,0,0,0,1.999836e+05,0.784292
2,train_buildings,707726,0,0,0,0,7.894039e+05,1.115409
3,test_buildings,184155,0,0,0,0,1.502439e+05,0.815856
4,train_contents,721617,0,0,0,0,8.107226e+05,1.123480
5,test_contents,187498,0,0,0,0,1.501238e+05,0.800669


Merge customer and adress info.
Objective: Enrich policy-level features before building the target.
These are many-to-one joins: each policy-period belongs to one customer and references one insured address.

In [34]:
def enrich_static_features(df):
    initial_rows = len(df)

    enriched = df.merge(
        customers,
        on="customer_id",
        how="left",
        validate="many_to_one"
    )

    enriched = enriched.merge(
        addresses,
        on="address_id",
        how="left",
        validate="many_to_one"
    )

    assert len(enriched) == initial_rows

    return enriched


train = {
    line: enrich_static_features(df)
    for line, df in train.items()
}

test = {
    line: enrich_static_features(df)
    for line, df in test.items()
}

Step 4 — Engineer historical weather features
Objective: Add weather risk information without using future observations.
We should not use actual weather from 2025 to predict 2025 risk premium. Equally, for a 2023 policy we should not use actual weather outcomes from 2023 when simulating a prediction made at inception.
A starting approach is to use previous-year weather.

In [38]:
# Identify weather variables
weather_columns = [
    col for col in weather.columns
    if col not in ["address_id", "year"]
]

# Rename weather columns
weather_lagged = weather.rename(
    columns={
        "year": "previous_year",
        **{
            col: f"lag1_{col}"
            for col in weather_columns
        }
    }
)

# Function to merge historical weather
def merge_weather(df):

    result = df.copy()

    # Derive year from the policy-period start date
    result["period_start"] = pd.to_datetime(
        result["period_start"]
    )

    result["year"] = result["period_start"].dt.year

    # Use weather from the previous calendar year
    result["previous_year"] = result["year"] - 1

    original_rows = len(result)

    # Merge weather
    result = result.merge(
        weather_lagged,
        on=["address_id", "previous_year"],
        how="left",
        validate="many_to_one"
    )

    # Check that the merge did not duplicate rows
    assert len(result) == original_rows

    return result.drop(columns="previous_year")


# Apply to training datasets
train = {
    line: merge_weather(df)
    for line, df in train.items()
}

# Apply to test datasets
test = {
    line: merge_weather(df)
    for line, df in test.items()
}

Step 5 — Match claims to the correct policy-period
Objective: Assign every historical claim to the exact row covering its loss date.
This is the most important merge in the workflow.
We cannot join claims only on policy_id, because a policy may have multiple rows across different years or mid-term changes.
First, combine the policy-period identifiers into a temporary lookup table.

In [39]:
policy_windows = pd.concat(
    [
        datasets[f"policies_{line}"][
            ["row_id", "policy_id", "period_start", "period_end"]
        ].assign(line=line)
        for line in ["motor", "buildings", "contents"]
    ],
    ignore_index=True
)

policy_windows["period_start"] = pd.to_datetime(
    policy_windows["period_start"]
)
policy_windows["period_end"] = pd.to_datetime(
    policy_windows["period_end"]
)

claims["loss_date"] = pd.to_datetime(claims["loss_date"])

claims_matched = claims.merge(
    policy_windows,
    on=["policy_id", "line"],
    how="left"
)

claims_matched = claims_matched.loc[
    (claims_matched["loss_date"] >= claims_matched["period_start"])
    &
    (claims_matched["loss_date"] <= claims_matched["period_end"])
].copy()

In [40]:
print("Original claims:", len(claims))
print("Matched claims:", len(claims_matched))

print(
    "Claims with multiple matches:",
    claims_matched["claim_id"].duplicated().sum()
)

assert claims_matched["claim_id"].is_unique

unmatched_claims = set(claims["claim_id"]) - set(
    claims_matched["claim_id"]
)

print("Unmatched claims:", len(unmatched_claims))

Original claims: 163150
Matched claims: 162736
Claims with multiple matches: 0
Unmatched claims: 414


### To do

In [53]:
print(unmatched_claims)

{'CL98432802', 'CL99142648', 'CL15443644', 'CL57052267', 'CL79269255', 'CL86729108', 'CL26515713', 'CL64761695', 'CL82012970', 'CL35427561', 'CL62305722', 'CL41823339', 'CL43855914', 'CL68891652', 'CL57869884', 'CL73815419', 'CL62134577', 'CL85454617', 'CL19693592', 'CL89591944', 'CL71489399', 'CL60380514', 'CL21044369', 'CL38716632', 'CL77668398', 'CL25217808', 'CL01789760', 'CL61630129', 'CL77215857', 'CL39197013', 'CL59961713', 'CL73133126', 'CL51924821', 'CL85052890', 'CL45624719', 'CL18522494', 'CL62419639', 'CL83780728', 'CL25802973', 'CL03692884', 'CL97697798', 'CL03458181', 'CL43328527', 'CL00112682', 'CL30154562', 'CL67297323', 'CL97175537', 'CL45634360', 'CL81279630', 'CL02340257', 'CL37364839', 'CL39785970', 'CL41570487', 'CL24898060', 'CL80859735', 'CL16772078', 'CL87637346', 'CL07716363', 'CL54445727', 'CL90213279', 'CL47614338', 'CL66433128', 'CL28118513', 'CL02009192', 'CL26373584', 'CL43259760', 'CL23393045', 'CL30227696', 'CL35765054', 'CL49721640', 'CL18429232', 'CL55

Step 6 — Calculate the observed net incurred claim cost
Objective: Construct a historical cost measure consistent with the target.
According to the data dictionary:
- paid_to_date is already net of excess.
- incurred_to_date consists of paid amounts and the open-claim case estimate.
- recoveries is a separate amount, but it is not confirmed whether incurred already reflects it.
- Repudiated claims must be excluded.
Our provisional assumption is:
\[
\text{Observed Net Incurred}
=
\text{Incurred to Date}
-
\text{Recoveries}
\]

We must not deduct the policy excess again.

In [54]:
claims_matched["eligible"] = (
    claims_matched["status"] != "repudiated"
)

claims_matched["recoveries"] = (
    claims_matched["recoveries"].fillna(0)
)

# Working assumption: recoveries are NOT yet deducted
# from incurred_to_date.
RECOVERIES_ALREADY_NETTED = False

if RECOVERIES_ALREADY_NETTED:
    claims_matched["net_cost_before_status"] = (
        claims_matched["incurred_to_date"]
    )
else:
    claims_matched["net_cost_before_status"] = (
        claims_matched["incurred_to_date"]
        - claims_matched["recoveries"]
    )

claims_matched["observed_net_incurred"] = np.where(
    claims_matched["eligible"],
    claims_matched["net_cost_before_status"],
    0
)

# Final reconciliation:
display(
    claims_matched.groupby(["line", "status"]).agg(
        claims=("claim_id", "count"),
        incurred=("incurred_to_date", "sum"),
        recoveries=("recoveries", "sum"),
        net_cost=("observed_net_incurred", "sum")
    )
)

negative_claims = claims_matched[
    claims_matched["observed_net_incurred"] < 0
]

print("Negative net incurred claims:", len(negative_claims))

claims      incurred   recoveries      net_cost
line      status                                                     
buildings closed       24485  2.165655e+08         0.00  2.165655e+08
          open          1890  3.120311e+07         0.00  3.120311e+07
contents  closed       22944  6.249864e+07         0.00  6.249864e+07
          open           809  3.467941e+06         0.00  3.467941e+06
          repudiated     299  0.000000e+00         0.00  0.000000e+00
motor     closed      103468  3.762674e+08  27613464.36  3.486539e+08
          open          7433  8.194159e+07         0.00  8.194159e+07
          repudiated    1408  0.000000e+00         0.00  0.000000e+00

Negative net incurred claims: 0


In [55]:
## check the AI code, if the sums of columns are correct, so that sum ultimate - recoveries is correct. 

Step 7 — Aggregate claims at policy-period level
Objective: Construct the two historical responses: N and total observed claim cost.
For each policy-period:
\[
N_i=\text{Number of eligible claims}
\]

\[
C_i=\sum_{j\in i}\text{Observed Net Incurred}_{ij}
\]

In [66]:
targets = (
    claims_matched.groupby("row_id")
    .agg(
        N=("eligible", "sum"),
        observed_net_incurred=(
            "observed_net_incurred", "sum"
        )
    )
    .reset_index()
)

targets["N"] = targets["N"].astype(int)

In [67]:
for line in train:
    initial_rows = len(train[line])

    train[line] = train[line].merge(
        targets,
        on="row_id",
        how="left",
        validate="one_to_one"
    )

    assert len(train[line]) == initial_rows

    train[line]["N"] = (
        train[line]["N"].fillna(0).astype(int)
    )

    train[line]["observed_net_incurred"] = (
        train[line]["observed_net_incurred"].fillna(0)
    )

MergeError: Passing 'suffixes' which cause duplicate columns {'N_y', 'N_x', 'observed_net_incurred_y', 'observed_net_incurred_x'} is not allowed.

Step 8 — Assess claim development to ultimate
Objective: Correct the mismatch between observed incurred and ultimate costs.
At 31 December 2024:
- Some 2024 losses are not yet reported (IBNR).
- Some reported claims remain open and may develop.
- Future recoveries may not yet be reflected.
Therefore, modelling observed_net_incurred directly will generally underestimate ultimate loss costs if those effects are material.
Ideally, use historical valuation snapshots to build paid/incurred development triangles. However, the supplied claims table has only one snapshot, so reliable chain-ladder development factors cannot be estimated from this table alone.
If validated development-to-ultimate factors become available, you can apply them at a suitable line/peril/accident-year level.

In [ ]:
# No estimated development factors are supplied yet.
# These should NOT be treated as genuine ultimate estimates.

for line in train:
    train[line]["development_factor"] = 1.0

    train[line]["model_cost_target"] = (
        train[line]["observed_net_incurred"]
        * train[line]["development_factor"]
    )

## EDA

In [ ]:
for name, df in datasets.items():
    print(f"\n{name} — first 5 rows")
    display(df.head())